# 「THE PRICE IS RIGHT」Capstone 專案

本週——根據 Amazon 爬蟲資料，從商品描述預測價格

# 進度安排

DAY 1：資料策展  
DAY 2：資料前處理  
DAY 3：評估、基線、傳統 ML  
DAY 4：深度學習與 LLM  
DAY 5：微調 Frontier 模型  

## DAY 3：評估、基線、傳統 ML

今天會寫簡單模型預測商品價格

我們會用一種方法評估模型效能

並用傳統機器學習測試一些基線模型

In [ ]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [ ]:
LITE_MODE = False

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
random.seed(42)
evaluate(random_pricer, test)

In [ ]:
# 挺有趣的！
# 我們可以做得更好——這是另一個相當簡單的模型

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

In [ ]:
evaluate(constant_pricer, test)

In [ ]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [ ]:
# 傳統線性迴歸！

np.random.seed(42)

# 分離特徵與目標
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# 訓練線性迴歸
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# 對測試集預測並評估
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [ ]:
evaluate(linear_regression_pricer, test)

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [ ]:
# 以下是它挑出的最常見詞彙（不含 stop words）：

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

## 隨機森林模型

隨機森林是一種「**ensemble**」（集成）演算法，結合多個較小演算法以做出更好預測。

它使用非常簡單的機器學習演算法——**決策樹**。決策樹透過檢查輸入特徵值做預測，像 IF 流程圖。決策樹很快很簡單，但容易過擬合。

在我們的案例中，「特徵」是向量的元素——也就是某詞在商品描述中出現的次數。

可這樣理解：

**決策樹**  
\- 若「TV」出現超過 3 次，則  
-- 若「LED」出現超過 2 次，則  
--- 若「HD」至少出現 1 次，則  
---- 價格 = $500


隨機森林會建立多棵決策樹。每棵用不同的隨機資料子集與特徵子集訓練。上面我們指定 100 棵樹，這是預設值。

然後隨機森林對所有樹的結果取平均，得到最終預測。

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)

In [ ]:
# 若要儲存模型（尤其在更大資料集上訓練時），可這樣做

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## 介紹 XGBoost

與隨機森林類似，XGBoost 也是結合多棵決策樹的集成模型。

但不同於隨機森林，XGBoost 一棵接一棵建立，每棵修正前面樹的錯誤，使用「梯度下降」。

它比隨機森林快得多，可跑完整資料集，且通常泛化更好。

**若 import 失敗請略過！非必要。在 Mac 上可能需在終端機執行 `brew install libomp`。**

In [ ]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
evaluate(xg_boost, test)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商業應用</h2>
            <span style="color:#181;">傳統 ML 不只適合學歷史；今天在業界仍大量使用，特別是特徵明確的任務。值得花時間探索演算法並實驗。看你能不能用我的傳統 ML 數字打敗我！我對完整 80 萬筆訓練集跑隨機森林，約 15 小時，最終誤差低至 $56.40。傳統 ML 可以做得很好——親自試試！</span>
        </td>
    </tr>
</table>